# 第4章 优化算法 · 教学练习



本册为依据课程材料新编的训练题，不是教师公布的考试原题。

## 材料依据
- [[课程/深度学习/PPT/Chapter4/Chapter4.md]]（最速下降、线搜索、Momentum/NAG、矩阵AdaGrad、RMSProp、Adam、AdaMax、小批量）
- [[课程/深度学习/课堂笔记/2026-09-16-第6节-深度学习优化理论讲座.md]]（第四至八节与学习率边界校注）
- [[课程/深度学习/课堂笔记/2026-09-21-第7节-随机优化算法讲解.md]]（第二至五节；Chapter4(1).py程序解读）
- [[课程/深度学习/演示代码/Chapter4(1).py]]（d=5、SGD/Adam/RMSprop、1000步、初始全1）
- [[课程/深度学习/作业/Homework02/Homework02.md]]（Exercise 5、6、8；原参数下如实报告发散）
- [[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]]（C4-03—10、A-06、07、10）
- [[课程/深度学习/PPT/2026深度学习第一次上机考试规则与命令表v1.1 (1)/2026深度学习第一次上机考试规则与命令表v1.1 (1).md]]（30分钟、100分及API范围）


## 使用方式

先运行环境单元，再逐题阅读、实现和自检。教学题允许提示与接口骨架；遇到“尚未完成”不代表通过。每题的代码需写在该题作答单元，答案在独立文件夹。学习时间不受 30 分钟限制；综合作业扩展题可分次完成。


In [ ]:
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
from pathlib import Path

# CPU 默认即可；不更改全局默认设备，也不依赖 CUDA、网络或额外数据文件。
torch.set_num_threads(1)
np.set_printoptions(precision=6, suppress=True)
torch.manual_seed(2026)
print("NumPy", np.__version__, "PyTorch", torch.__version__)


## Q401 驻点、鞍点与固定学习率的临界边界

**新编代码化，依据 Homework02 Exercise 5、Chapter4学习率例子、汇编 C4-03。**

1. 对 $s(x)=8x_1+12x_2+x_1^2-2x_2^2$ 求唯一驻点，沿两坐标微扰验证一增一减，不能把“梯度为0”当极小值。
2. 对 $q(x)=x_1^2+4x_2^2$，实现 `quadratic_gd(eta,steps)`，初值 `(1,1)`，返回包括初值的点序列。
比较学习率 `.1,.2,.25,.3` 的20步损失，保存 `output/Q401.png`（对数纵轴需正数）。打印第一步，解释振荡、临界不衰减、发散。


### 自检接口／需保存的结果

保存 `stationary401`（唯一驻点，NumPy数组，形状 `(2,)`）及 `changes401`（形状 `(2,)`）：取正微扰量 `.01`，两项分别记录从驻点沿第1、第2坐标正向微扰后，函数值相对驻点函数值的变化，不是微扰后的函数值本身。

`quadratic_gd(eta,steps)` 返回NumPy点序列，形状 `(steps+1,2)`，第0行为初值，其余行按更新次序排列。图片名称沿用题面，无须保存各学习率路径的额外全局名。


<details><summary>提示（先尝试再展开）</summary>

两个坐标的迭代乘子分别为 `1-2*eta`、`1-8*eta`；收敛要求两者绝对值都小于1。


</details>


In [ ]:
# TODO：在此实现本题，再运行下一自检单元。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
try:
    assert np.allclose(stationary401,[-4,3]) and changes401[0]>0 and changes401[1]<0
    assert np.allclose(quadratic_gd(.2,1)[1],[.6,-.6])
    assert np.allclose(np.abs(quadratic_gd(.25,20)[:,1]),1)
    assert np.linalg.norm(quadratic_gd(.3,20)[-1])>100
    assert np.linalg.norm(quadratic_gd(.1,20)[-1])<.02
    assert Path('output/Q401.png').stat().st_size>1000
except (NameError, NotImplementedError) as exc:
    print("尚未完成本题：", exc)
else:
    print("自检通过；仍需人工核对题目全部要求。")


## Q402 线搜索不是只解一个驻点方程

**Homework02 Exercise 6原数值＋新编验证。**

$f(x)=(x_1+x_2^2)^2$，初值 `(0,1)`，沿负梯度方向作精确线搜索。
手动求梯度与一元函数 $g(\eta)=f(x-\eta\nabla f(x))$；找出所有正的全局最优步长 `optimal_steps402`，而不是只报一个驻点。
用 Python 计算每个最优步长的新点与目标值，验证 `eta=5/16` 虽然是驻点却非最优。
再对 $q(x)=x_1^2+4x_2^2$ 实现 `quadratic_line_step(x)`，用闭式最优步长计算一步，核对新梯度与旧搜索方向正交。


### 自检接口／需保存的结果

- 定义 `objective402(x)`：输入形状 `(2,)` 的NumPy点，返回题面第一个目标f的标量值；保存初始点 `x402` 和该点解析梯度 `gradient402`，均为形状 `(2,)` 的NumPy数组。
- `optimal_steps402` 为全部正全局最优步长组成的一维NumPy数组，按升序排列；`middle_value402` 为沿该初始点负梯度方向、步长 `5/16` 处的目标值（标量）。
- `quadratic_line_step(x)` 按 `(new_point,eta)` 返回新点与步长，分别为形状 `(2,)` 的NumPy数组和标量；在零梯度处返回原点副本与步长0。对二次目标、初值 `[1,1]` 运行的两项结果，分别保存为 `new402,eta402`。


<details><summary>提示（先尝试再展开）</summary>

沿搜索方向内层为 `16*eta**2-10*eta+1`，平方后目标非负，内层的两个正根都达到全局最小值0。


</details>


In [ ]:
# TODO：在此实现本题，再运行下一自检单元。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
try:
    assert np.allclose(optimal_steps402,[.125,.5])
    assert all(abs(objective402(x402-eta*gradient402))<1e-14 for eta in optimal_steps402)
    assert middle_value402>0
    assert abs(eta402-17/130)<1e-12
    assert abs(np.array([2.,8.])@np.array([2*new402[0],8*new402[1]]))<1e-12
    assert quadratic_line_step(np.zeros(2))[1]==0
except (NameError, NotImplementedError) as exc:
    print("尚未完成本题：", exc)
else:
    print("自检通过；仍需人工核对题目全部要求。")


## Q403 Momentum 与 NAG：先看清速度的约定

**新编，依据 Chapter4“动量法”、汇编 C4-05。**

在 $q(x)=x_1^2+4x_2^2$ 上手动实现 `momentum_run(x,eta,mu,steps,nag=False)`，返回点与速度历史（均含初始状态）。
采用课件速度约定 $v_{t}=\mu v_{t-1}-\eta g$、$x_t=x_{t-1}+v_t$；初速度0。
NAG 的梯度在 `x+mu*v` 评价。固定初值 `(1,1)`、eta=.1、mu=.5，比较前两步，不能把动量误写成直接加正梯度。


### 自检接口／需保存的结果

`momentum_run(...)` 按 `(points,velocities)` 返回两个NumPy数组，形状都为 `(steps+1,2)`；第0行分别是初始点与零速度。

对题面指定初值、eta、mu运行两步：常规动量的点历史保存为 `mom403`，速度历史保存为 `vm403`；相同条件下NAG的点历史保存为 `nag403`，三者形状均为 `(3,2)`。NAG速度历史的变量名可自行安排。


<details><summary>提示（先尝试再展开）</summary>

第一步两种方法相同；第二步NAG的预看位置是 `(.7,-.2)`，不是当前 `(.8,.2)`。


</details>


In [ ]:
# TODO：在此实现本题，再运行下一自检单元。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
try:
    assert np.allclose(mom403[1],[.8,.2]) and np.allclose(vm403[1],[-.2,-.8])
    assert np.allclose(mom403[2],[.54,-.36])
    assert np.allclose(nag403[2],[.56,-.04])
    p,v=momentum_run([1.,1.],.1,0,2)
    assert np.allclose(p,[[1,1],[.8,.2],[.64,.04]])
except (NameError, NotImplementedError) as exc:
    print("尚未完成本题：", exc)
else:
    print("自检通过；仍需人工核对题目全部要求。")


## Q404 AdaGrad：课件全矩阵形式不等于逐坐标实现

**新编，依据 Chapter4“AdaGrad算法”、9月16日第七节、汇编 C4-06。**

课件为 $G_t=\sum_{s=1}^tg_sg_s^T$、$x_t=x_{t-1}-\eta G_t^{-1/2}g_t$，奇异时取逆平方根的伪逆。
实现 `matrix_adagrad_step(x,g,G,eta)` 返回新点和新G。
用对称特征分解，特征值大于 `1e-12*max(1,max(abs(eigenvalues)))` 时取倒数平方根，其余置0。
**扩展API**：`np.linalg.eigh`、布尔索引；不属于命令表中要求背记的基础命令。

初值 `(1,1)`、梯度 `(3,4)`、历史零矩阵、eta=.1，输出一步结果，并与逐坐标AdaGrad的 `(0.9,0.9)` 比较。
再输入第二个非平行梯度 `(4,-3)`，验证累计G；最后对零梯度做无NaN检查。


### 自检接口／需保存的结果

`matrix_adagrad_step(...)` 按 `(xnew,Gnew)` 返回，类型均为NumPy数组，形状分别为 `(2,),(2,2)`。题面首步结果保存为 `x404,G404`；从该首步结果接着输入第二个指定梯度，结果保存为 `x2404,G2404`，形状对应不变。自检另行创建零梯度输入，其他内部特征分解变量可自定名。


<details><summary>提示（先尝试再展开）</summary>

首步G秩1；逆平方根作用于g得到 `g/||g||=(.6,.8)`，不是分别得到 `(1,1)`。


</details>


In [ ]:
# TODO：在此实现本题，再运行下一自检单元。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
try:
    assert np.allclose(x404,[.94,.92])
    assert np.allclose(G404,[[9,12],[12,16]])
    assert np.allclose(G2404,25*np.eye(2)) and np.allclose(x2404,[.86,.98])
    xz,Gz=matrix_adagrad_step(np.array([1.,2.]),np.zeros(2),np.zeros((2,2)),.1)
    assert np.allclose(xz,[1,2]) and np.all(np.isfinite(xz))
    assert not np.allclose(x404,[.9,.9])
except (NameError, NotImplementedError) as exc:
    print("尚未完成本题：", exc)
else:
    print("自检通过；仍需人工核对题目全部要求。")


## Q405 RMSProp：指数平方平均与零分母

**新编，依据 Chapter4“RMSProp算法”、Homework02 Exercise 8参数。**

实现 `rmsprop_step(x,g,v,eta=.01,gamma=.1)`，返回 `(xnew,vnew)`，严格用
$v_t=\gamma v_{t-1}+(1-\gamma)g_t^2$，$x_t=x_{t-1}-\eta g_t/\sqrt{v_t}$。
课件未给epsilon；本题不加epsilon，对 `g=0,v=0` 的通道约定更新为0（不计算0/0）。
从 `x=[1,1],v=[0,0]` 依次输入梯度 `[3,-4]`、`[0,2]`，打印两步。不要把gamma与新梯度权重弄反。
扩展API：`np.divide(...,out=...,where=...)` 用于安全零除；可用循环/掩码代替。


### 自检接口／需保存的结果

`rmsprop_step(...)` 按 `(xnew,vnew)` 返回两个NumPy数组，形状均为 `(2,)`；不传可选参数时使用题面默认的eta与gamma。首步结果保存为 `x405,v405`；将该首步点与历史量传入下一步，第二步结果保存为 `x2405,v2405`，四者形状均为 `(2,)`。不要将两步结果保存成同一可变数组的共享引用。


<details><summary>提示（先尝试再展开）</summary>

第一个平方平均为 `.9*g*g`；零梯度不一定让历史v归零，但当前参数更新仍为0。


</details>


In [ ]:
# TODO：在此实现本题，再运行下一自检单元。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
try:
    assert np.allclose(v405,[8.1,14.4])
    assert np.allclose(x405,1-.01*np.array([1.,-1.])/np.sqrt(.9))
    assert np.allclose(v2405,[.81,5.04])
    assert abs(x2405[0]-x405[0])<1e-12 and abs(x2405[1]-(x405[1]-.02/np.sqrt(5.04)))<1e-12
    xz,vz=rmsprop_step(np.array([1.,2.]),np.zeros(2),np.zeros(2))
    assert np.allclose(xz,[1,2]) and np.all(np.isfinite(xz))
except (NameError, NotImplementedError) as exc:
    print("尚未完成本题：", exc)
else:
    print("自检通过；仍需人工核对题目全部要求。")


## Q406 Adam 与 AdaMax：首步偏差修正逐项校准

**新编，依据 Chapter4“Adam”“AdaMax”、汇编 C4-07。**

实现 `optimizer_step(x,g,state,name,...)` 中Adam与AdaMax（也可实现全部6种，供长作业调用），时间计数从1开始，零初始状态。
Adam：$m_t=\beta_1m_{t-1}+(1-\beta_1)g_t$，$v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2$，
$\hat m=m_t/(1-\beta_1^t),\hat v=v_t/(1-\beta_2^t)$，更新 $-\eta\hat m/(\sqrt{\hat v}+\epsilon)$。
AdaMax：$u_t=\max(\beta_2u_{t-1},|g_t|)$，更新 $-\eta m_t/((1-\beta_1^t)u_t)$；**u没有二阶偏差修正**。
AdaMax按课件不加eps，零u通道约定零更新。

设置 `eta=.01,beta1=.9,beta2=.999,eps=.001`，用初始 `[1,1]`、首梯度 `[3,-4]`、第二梯度 `[-1,2]` 验证两步。
保存 `adam_first406,adam_second406,adam_state406` 和 `amax_first406,amax_second406,amax_state406`。
本题完整6种参考实现包含特征分解等扩展API；只做Adam/AdaMax时无须实现其余分支。

### 自检接口／需保存的结果

函数可采用完整签名 `optimizer_step(x,g,state,name,eta=.01,mu=.1,gamma=.1,beta1=.9,beta2=.999,eps=.001)`；本题自检只调用名称 `Adam,AdaMax`。返回值为更新后的NumPy点，形状 `(2,)`；传入的空字典 `state` 须就地初始化，并在后续调用中保存同一次运行的状态。

已指定的 `adam_first406,adam_second406,amax_first406,amax_second406` 都是各步点的独立数值快照，形状 `(2,)`。`adam_state406` 保存第二步完成后的状态，至少含键 `m,v`（各为形状 `(2,)` 的NumPy数组）；`amax_state406` 至少含键 `m,u`（同形状）。两个方法使用独立字典，额外计时或缓存键可自定。


<details><summary>提示（先尝试再展开）</summary>

首步修正后mhat=g、vhat=g²；Adam首步是 `-eta*g/(abs(g)+eps)`。AdaMax首步非零通道恰为 `-eta*sign(g)`。


</details>


In [ ]:
# TODO：在此实现本题，再运行下一自检单元。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
try:
    assert np.allclose(adam_first406,1-.01*np.array([3.,-4.])/(np.array([3.,4.])+.001))
    assert np.allclose(amax_first406,[.99,1.01])
    m2=.9*(.1*np.array([3.,-4.]))+.1*np.array([-1.,2.])
    v2=.999*(.001*np.array([9.,16.]))+.001*np.array([1.,4.])
    u2=np.maximum(.999*np.array([3.,4.]),[1.,2.])
    assert np.allclose(adam_state406['m'],m2) and np.allclose(adam_state406['v'],v2)
    assert np.allclose(adam_second406,adam_first406-.01*(m2/(1-.9**2))/(np.sqrt(v2/(1-.999**2))+.001))
    assert np.allclose(amax_state406['u'],u2)
    assert np.allclose(amax_second406,amax_first406-.01*m2/((1-.9**2)*u2))
    for name in ['Adam','AdaMax']:
        xz=optimizer_step(np.array([1.,2.]),np.zeros(2),{},name)
        assert np.allclose(xz,[1,2]) and np.all(np.isfinite(xz))
except (NameError, NotImplementedError) as exc:
    print("尚未完成本题：", exc)
else:
    print("自检通过；仍需人工核对题目全部要求。")


## Q407 复现 Chapter4(1).py：三优化器的完整数值日志

**原课堂演示独立复现，非新造教师参数。**

保留 `d=5`、初始全1、学习率.01、1000次更新，分别用SGD、Adam、RMSprop。
目标用Python零起始下标写为
$F(x)=\frac1{d-3}\sum_{i=1}^{d-3}(x_{i-1}x_i+x_ix_{i+1}+x_{i+1}x_{i+2}-i/(d-3))^2$。
实现 `demo_objective(x)` 与 `run_demo_optimizer(name)`，返回最终参数、1001个**更新后重算**的目标值。
每种方法必须重建参数和优化器，保存 `demo_results407`，画半对数曲线 `output/Q407.png`。

本练习为数值校准将张量显式设为float64（原脚本默认float32），其它设置不改；库算法参数采用各自默认值。
教学扩展：`torch.optim.RMSprop` 不在考试命令表核心两优化器之列；本题为课堂覆盖练习，不强加到测试。
说明：这里没有任何mini-batch采样；优化器名叫SGD不代表这个程序使用随机梯度。

### 自检接口／需保存的结果

`demo_objective(x)` 接收一维PyTorch张量并返回可反传的标量张量。`run_demo_optimizer(name)` 按 `(final_x,history)` 返回两个NumPy数组，形状分别为 `(5,),(1001,)`，history包含初值及1000次更新后的目标值。

`demo_results407` 保存为字典，键准确使用 `SGD,Adam,RMSprop`，每个值均为该方法返回的二元组；日志应是纯数值，不是计算图对象。图片文件名沿用题面。


<details><summary>提示（先尝试再展开）</summary>

初值每个括号分别是2.5、2，故初始目标是41/8。目标中i=1时实际访问x的0—3号分量。


</details>


In [ ]:
# TODO：在此实现本题，再运行下一自检单元。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
try:
    for name,(x,history) in demo_results407.items():
        assert history.shape==(1001,) and abs(history[0]-41/8)<1e-12
        assert np.all(np.isfinite(history)) and history[-1]<history[0]
        assert abs(history[-1]-demo_objective(torch.tensor(x)).item())<1e-12
    probe=torch.tensor([1.]*5,dtype=torch.float64,requires_grad=True)
    demo_objective(probe).backward()
    assert np.allclose(probe.grad.numpy(),[2.5,7.,9.,6.5,2.])
    assert Path('output/Q407.png').stat().st_size>1000
except (NameError, NotImplementedError) as exc:
    print("尚未完成本题：", exc)
else:
    print("自检通过；仍需人工核对题目全部要求。")


## Q408 小批量：末批、加权平均与真正的更新

**新编，依据9月21日第二节、Chapter4小批量、汇编 C4-08。**

数据 `x=linspace(-1,1,11)`、`y=2*x-.5`；线性模型 $a x+b$，损失MSE。
实现 `batch_gradient(theta,x,y)` 和 `make_batches(n,size,rng)`；批量大小4，打乱一次后必须覆盖所有11条且无重复，末批3条。
在**相同参数** `theta=[0,0]` 下，验证“各批平均梯度按批大小加权”恰等全梯度；未经加权的三批平均一般不等。
再做60轮小批量SGD（eta=.1），每轮重新打乱，保存 `theta408,losses408,steps408`（含初值的轮末全损失）。
不能声称更新中的三批梯度还可直接平均为同一个参数点的全梯度。
只需命令表的随机均匀数与 `np.argsort` 即可打乱，不要求DataLoader。


### 自检接口／需保存的结果

- `batch_gradient(theta,x,y)` 返回形状 `(2,)` 的NumPy平均梯度；`make_batches(n,size,rng)` 返回索引数组的list，每个元素是一批样本的整数索引。
- 将 `make_batches(11,4,np.random.default_rng(408))` 得到的首轮划分保存为 `batches408`，三批大小应依次为4、4、3。在同一参数 `[0,0]` 下，分别保存全数据梯度 `full_gradient408`、按批大小加权的批梯度平均 `weighted408`、未经加权的三批梯度平均 `unweighted408`，三者均为形状 `(2,)` 的NumPy数组；这些验证量须在参数更新前计算。
- 训练部分保存最终参数 `theta408`（NumPy数组，形状 `(2,)`）、轮末全损失 `losses408`（NumPy数组，形状 `(61,)`，含初始损失）、实际更新次数 `steps408`（整数）。训练打乱可另建seed=1408的生成器，避免首轮验证消耗的随机数改变训练实验。


<details><summary>提示（先尝试再展开）</summary>

最后一批较小时，等权平均批梯度相当于过度加权该批样本。一个epoch为3次更新，60轮是180步。


</details>


In [ ]:
# TODO：在此实现本题，再运行下一自检单元。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
try:
    assert [len(ix) for ix in batches408]==[4,4,3]
    assert np.array_equal(np.sort(np.concatenate(batches408)),np.arange(11))
    assert np.allclose(full_gradient408,[-1.6,1.]) and np.allclose(weighted408,full_gradient408)
    assert not np.allclose(unweighted408,full_gradient408)
    assert steps408==180 and losses408.shape==(61,) and losses408[-1]<1e-9
    assert np.max(np.abs(theta408-[2.,-.5]))<1e-4
except (NameError, NotImplementedError) as exc:
    print("尚未完成本题：", exc)
else:
    print("自检通过；仍需人工核对题目全部要求。")


## Q409 Homework02 Exercise 8：六手动优化器×四初值，诚实记录发散

**原作业核心编程任务完整覆盖，长作业扩展，建议60—90分钟；不是30分钟测试。**

$f(x)=100(x_2-x_1^2)^2+(1-x_1)^2$，全局极小点 `(1,1)`、最小值0。
手动实现GD、Momentum、**课件全矩阵AdaGrad**、RMSProp、Adam、AdaMax，禁止调用torch.optim代替。
四个初值依次 `(1,0),(0,1),(-1,-1),(3,3)`；每组预算1000步。**保留原作业参数**：

| 方法 | 参数 |
|---|---|
| GD | eta=.01 |
| Momentum | mu=.1, eta=.01；速度已含负学习率，v0=0 |
| AdaGrad | eta=.01；G0=0，全矩阵逆平方根伪逆 |
| RMSProp | gamma=.1, eta=.01，v0=0 |
| Adam | beta1=.9,beta2=.999,eta=.01,eps=.001，两个偏差修正 |
| AdaMax | beta1=.9,beta2=.999,eta=.01，只有一阶矩偏差修正 |

课件RMSProp/AdaMax未给eps，采用零分母通道零更新；AdaGrad按Q404的特征阈值策略。
这几个**数值实现约定**须注明，不能偷偷改eta或使用梯度裁剪后宣称“原设定收敛”。

编写 `rosenbrock`、`rosenbrock_grad`、`run_rosenbrock(name,x0,steps=1000)`；字典结果至少含
`status,stop_step,completed_steps,last_finite_x,final_x,distance,final_loss,losses,points`。
若出现非有限梯度/新点/目标，立即停止该次运行，标为发散，`final_x=None`、距离和最终目标记inf，
不能把最后有限点冒充1000步答案。正常走满预算标记 `completed_budget`，**这不等于收敛**。
保存24个结果于 `results409`，输出状态表、真实迭代数、最终点/最后有限点、距 `(1,1)` 的误差；
每个初值画两个视图：前12步的6条有效前缀，以及走满预算方法的1000步曲线，写入 `output/Q409.png`。
这样既保留发散到非有限前的真实大值，又不会让它把自适应方法的低损失变化压成一条线；完整数值仍以上述表格为准。

扩展API：`np.linalg.eigh,isfinite,errstate,divide(where=...)`、`plt.subplots`，均为长作业数值保护/显示工具，不在测试中要求默写。
另做六种算法各自的**一步数值校准**；不能只验证曲线形状。


### 自检接口／需保存的结果

`rosenbrock(x)` 返回标量目标值，`rosenbrock_grad(x)` 返回形状 `(2,)` 的NumPy梯度；输入x为长度2的NumPy点。`results409` 为含24个字典的list，每个字典对应一种方法与一个初值组合；`run_rosenbrock(...)` 的返回字典使用以下完整结构：

| 键 | 类型与含义 |
|---|---|
| `method` | 方法名字符串，使用 `GD,Momentum,AdaGrad,RMSProp,Adam,AdaMax` 之一 |
| `initial` | 初始点的独立NumPy数组，形状 `(2,)` |
| `status` | 正常用满预算为 `completed_budget`；非有限退出用以 `diverged` 开头的原因字符串 |
| `stop_step` | 正常完成为 `None`；发散时为首次失败尝试的步号，采用1起始计数 |
| `completed_steps` | 已成功接受的有限更新次数，整数；不包含失败的那次尝试 |
| `last_finite_x` | 最后一个有限点，NumPy数组，形状 `(2,)` |
| `final_x` | 正常完成时为最终点的NumPy数组，形状 `(2,)`；发散时为 `None` |
| `distance` | 正常完成时最终点到题面参考解的欧氏距离；发散时为浮点inf |
| `final_loss` | 正常完成时在最终点重算的目标值；发散时为浮点inf |
| `losses` | NumPy数值数组，形状 `(completed_steps+1,)`；第0项是初始目标，仅保留有限值 |
| `points` | NumPy数组，形状 `(completed_steps+1,2)`；第0行为初始点，逐行对应losses |

另保存字典 `one_step409`，键为上述六个方法名，每个值为形状 `(2,)` 的一步更新后NumPy点。此一步校准统一使用起点 `[1,1]`、给定梯度 `[3,-4]`、每种方法独立的零初始状态，以及题面表格中的原超参数；这里给定的是梯度输入，不再把它解释为Rosenbrock在该点的梯度。其余绘图、状态缓存变量可自定。


<details><summary>提示（先尝试再展开）</summary>

先校准梯度 `(1,0)→(400,-200)`：GD原步长首步立即到 `(-3,2)`，目标会增大。发散是应报告的数值结果，不一定是公式编错。


</details>


In [ ]:
# TODO：在此实现本题，再运行下一自检单元。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
try:
    assert rosenbrock(np.ones(2))==0 and np.allclose(rosenbrock_grad(np.ones(2)),[0,0])
    assert np.allclose(rosenbrock_grad(np.array([1.,0.])),[400,-200])
    probe=np.array([-.7,.4]);h=1e-6
    fd=np.array([(rosenbrock(probe+h*np.eye(2)[j])-rosenbrock(probe-h*np.eye(2)[j]))/(2*h) for j in range(2)])
    assert np.max(np.abs(fd-rosenbrock_grad(probe)))<1e-5
    expected={
     'GD':np.array([.97,1.04]),'Momentum':np.array([.97,1.04]),
     'AdaGrad':np.array([.994,1.008]),
     'RMSProp':1-.01*np.array([1.,-1.])/np.sqrt(.9),
     'Adam':1-.01*np.array([3.,-4.])/(np.array([3.,4.])+.001),
     'AdaMax':np.array([.99,1.01])}
    assert all(np.allclose(one_step409[name],target,atol=1e-12) for name,target in expected.items())
    assert len(results409)==24
    assert any(r['final_x'] is None for r in results409 if r['method']=='GD')
    for r in results409:
        assert len(r['losses'])==r['completed_steps']+1 and r['points'].shape==(len(r['losses']),2)
        assert np.all(np.isfinite(r['losses']))
        assert np.allclose(r['points'][0],r['initial'])
        assert abs(r['losses'][0]-rosenbrock(r['initial']))<1e-10
        if r['final_x'] is None:
            assert r['status'].startswith('diverged') and r['completed_steps']<1000
            assert np.isinf(r['distance']) and np.isinf(r['final_loss'])
        else:
            assert r['completed_steps']==1000 and r['status']=='completed_budget'
            assert abs(r['distance']-np.linalg.norm(r['final_x']-1))<1e-10
            assert abs(r['final_loss']-rosenbrock(r['final_x']))<1e-10
    assert Path('output/Q409.png').stat().st_size>1000
except (NameError, NotImplementedError) as exc:
    print("尚未完成本题：", exc)
else:
    print("自检通过；仍需人工核对题目全部要求。")


## Q410 梯度连乘与可控的稳定优化实验

**新编，依据 Chapter4梯度消失/爆炸、汇编 C4-09。**

1. 全部预激活为正、宽度1的ReLU链，所有隐藏及输出权重均为a、偏置0，输入x=1，隐藏层数L。
用自动微分实现 `chain_gradient(a,L)` 计算输出对输入的导数，比较 a=.5、1、2 与L=1,5,10,20，验证为 $a^{L+1}$，不用图像代替数值断言。
2. 另用稳定新编目标 $J(w)=(w_1-2)^2+2(w_2+1)^2$、初始 `(0,0)`，SGD学习率.1，100步，保存 `stable_history410,stable_w410`。
打印初末损失与最终参数，说明这不是将Rosenbrock原参数改小后冒称原作业收敛。

### 自检接口／需保存的结果

`chain_gradient(a,L)` 返回普通数值，含义为题面链式网络输出对标量输入的导数。`stable_w410` 保留100次更新后的CPU float64张量，形状 `(2,)`，使自检可读取其 `detach().numpy()`；`stable_history410` 为含101个普通数值的list或一维NumPy数组，依次保存初始损失及每次更新后的损失。其它局部变量无需固定命名。


<details><summary>提示（先尝试再展开）</summary>

所有ReLU输入为正，局部导数均为1，所以只剩权重连乘；注意总权重有L+1个而非L个。


</details>


In [ ]:
# TODO：在此实现本题，再运行下一自检单元。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
try:
    for a in [.5,1.,2.]:
        for L in [1,5,10,20]:
            assert abs(chain_gradient(a,L)-a**(L+1))<1e-12
    assert len(stable_history410)==101 and stable_history410[0]==6
    assert stable_history410[-1]<1e-16
    assert np.max(np.abs(stable_w410.detach().numpy()-[2.,-1.]))<1e-8
except (NameError, NotImplementedError) as exc:
    print("尚未完成本题：", exc)
else:
    print("自检通过；仍需人工核对题目全部要求。")
